# Week 1 — LLM Experimentation Notebook
### Tokenization, Embeddings, Semantic Search & API Parameter Experiments

**Instructions:**
- Sections 1–2 (Tokenization, Embeddings) run fully offline using open-source libraries — no API key needed.
- Section 3 (LLM API experimentation) requires an API key (Anthropic or OpenAI). Instructions are provided; skip/mock if you don't have one, and use a Playground UI instead (see exercises sheet).
- Fill in the `# TODO` cells yourself, run each cell, and write a short observation in the markdown cell provided after each exercise.


## Setup — Install Required Libraries

In [ ]:
# Run this once
!pip install tiktoken sentence-transformers scikit-learn numpy --quiet


## Section 1 — Tokenization

**Goal:** See how text gets broken into tokens, and how token count varies by content and model.


In [ ]:
import tiktoken

# GPT-style tokenizer (cl100k_base is used by GPT-3.5/4 family; good general-purpose example)
enc = tiktoken.get_encoding("cl100k_base")

sample_sentences = [
    "Generative AI is transforming how we build software.",
    "supercalifragilisticexpialidocious",
    "The transformer architecture uses self-attention.",
    "नमस्ते, आप कैसे हैं?",       # non-English example
    "def add(a, b):\n    return a + b"  # code example
]

for s in sample_sentences:
    tokens = enc.encode(s)
    print(f"Text: {s!r}")
    print(f"  Token count: {len(tokens)}")
    print(f"  Tokens (decoded individually): {[enc.decode([t]) for t in tokens]}")
    print()


**Exercise 1.1:** Add 3 of your own sentences to `sample_sentences` above (try one long technical sentence, one sentence with an uncommon/made-up word, and one in a language other than English). Rerun the cell.

**Reflection (write here):** Which sentence had the most tokens relative to its word count? Why do you think that happened? _(TODO: your answer)_


In [ ]:
# TODO: Exercise 1.2 — Compare token count vs. word count
# For each sentence in sample_sentences, print: word_count, token_count, and the ratio (tokens/word)

for s in sample_sentences:
    tokens = enc.encode(s)
    word_count = len(s.split())
    token_count = len(tokens)
    ratio = token_count / max(word_count, 1)
    print(f"{s[:40]!r:45} words={word_count:3}  tokens={token_count:3}  ratio={ratio:.2f}")


## Section 2 — Embeddings & Semantic Search

**Goal:** Generate embeddings for sentences and use cosine similarity to find semantically related sentences — the intuition behind semantic search (Week 2 preview).


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# A small, fast open-source embedding model — good for classroom use
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "I love hiking in the mountains.",
    "The stock market fell sharply today.",
    "My dog loves to play fetch in the park.",
    "Interest rates rose this quarter.",
    "We went camping last weekend near the lake."
]

embeddings = model.encode(sentences)
print("Embedding shape per sentence:", embeddings.shape)


In [ ]:
# Compute pairwise cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix:\n")
print("      " + "  ".join([f"S{i+1}" for i in range(len(sentences))]))
for i, row in enumerate(sim_matrix):
    print(f"S{i+1}  " + "  ".join([f"{v:.2f}" for v in row]))

print()
for i, s in enumerate(sentences):
    print(f"S{i+1}: {s}")


**Exercise 2.1 (Part H from exercises sheet):** Look at the similarity matrix above.
- Which pair of sentences has the highest similarity (excluding a sentence with itself)?
- Does this match your intuition? _(TODO: your answer)_

**Exercise 2.2:** Add 2 new sentences of your own — one that should be semantically close to an existing sentence, and one that should be far from all of them. Rerun Section 2 and confirm your prediction.


In [ ]:
# TODO: Exercise 2.2 — add your own sentences and rerun
my_sentences = sentences + [
    # "TODO: your sentence 1",
    # "TODO: your sentence 2",
]

my_embeddings = model.encode(my_sentences)
my_sim_matrix = cosine_similarity(my_embeddings)

for i, row in enumerate(my_sim_matrix):
    print(f"S{i+1}  " + "  ".join([f"{v:.2f}" for v in row]))


In [ ]:
# TODO: Exercise 2.3 — Simple semantic search function
# Given a query sentence, find the most similar sentence from `sentences` using cosine similarity.

def semantic_search(query, corpus, corpus_embeddings, top_k=1):
    query_embedding = model.encode([query])
    sims = cosine_similarity(query_embedding, corpus_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(corpus[i], sims[i]) for i in top_idx]

# Try it out
query = "outdoor adventure in nature"  # TODO: try your own queries too
results = semantic_search(query, sentences, embeddings, top_k=3)
for text, score in results:
    print(f"{score:.3f}  |  {text}")


**Reflection:** Try a query using a synonym or related concept that does NOT share exact keywords with any sentence (e.g., "financial markets" instead of "stock market"). Does semantic search still find the right sentence? Compare this to what a simple keyword search (`if word in sentence`) would have found. _(TODO: your answer)_


## Section 3 — LLM API Experimentation (Parameters)

**Goal:** Directly observe how `temperature`, `max_tokens`, and system prompts affect model output.

> **Note:** This section requires an API key. Set it as an environment variable before running (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`), or use the Anthropic Console / OpenAI Playground UI instead and log your observations manually in the exercises sheet.


In [ ]:
# Install the Anthropic SDK if using Claude
!pip install anthropic --quiet


In [ ]:
import os
from anthropic import Anthropic

# Make sure ANTHROPIC_API_KEY is set in your environment before running this cell
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

def ask_claude(prompt, system=None, temperature=1.0, max_tokens=200, model="claude-sonnet-4-6"):
    kwargs = dict(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text


### Exercise 3.1 — Temperature

Run the same creative prompt at three different temperatures. Compare creativity vs. consistency (run each setting 2–3 times).


In [ ]:
prompt = "Write a two-line opening sentence for a mystery novel."

for temp in [0.0, 0.7, 1.2]:
    print(f"--- Temperature = {temp} ---")
    for i in range(2):
        # TODO: uncomment once your API key is set
        # output = ask_claude(prompt, temperature=temp, max_tokens=60)
        # print(output)
        pass
    print()


**Reflection:** At temperature 0, did repeated runs give (near-)identical outputs? What changed as temperature increased? _(TODO: your answer)_


### Exercise 3.2 — Max Tokens

Ask for a detailed explanation with a very low `max_tokens` limit, then a higher one. Observe where the output gets cut off.


In [ ]:
prompt = "Explain how transformers use self-attention, in detail."

# TODO: uncomment once your API key is set
# short_output = ask_claude(prompt, max_tokens=20)
# long_output = ask_claude(prompt, max_tokens=300)
# print("SHORT (max_tokens=20):\n", short_output)
# print("\nLONG (max_tokens=300):\n", long_output)


### Exercise 3.3 — System Prompt vs. User Prompt

Set a system-level instruction and send several different user questions to see if it's consistently followed.


In [ ]:
system_instruction = "Always answer in exactly one sentence, no matter the question."

questions = [
    "What is a transformer model?",
    "Why do we need tokenization?",
    "What is the difference between pretraining and fine-tuning?",
]

for q in questions:
    # TODO: uncomment once your API key is set
    # output = ask_claude(q, system=system_instruction, max_tokens=100)
    # print(f"Q: {q}\nA: {output}\n")
    pass


**Reflection:** Was the system instruction followed consistently across all three questions? Note any cases where it was ignored or only partially followed. _(TODO: your answer)_

---

## Wrap-Up

Write a 3–5 sentence summary of what you learned this week about tokenization, embeddings, and LLM parameters, and one open question you still have.

_(TODO: your summary)_
